In [2]:

"""
CN7 03 - 원본 물리단위 기반 EDA
================================

전제
----
CN7_02에서 moldset_labeled_cn7 1211행의 원본 row mapping 복원이 완료되었다.

목표
----
1) mapping을 이용해 raw 물리단위 CN7 1211행을 재구성
2) PassOrFail 의미를 Reason과 함께 검증
3) Target / Side / 날짜 / SERIAL 구조 확인
4) 상수 / 저분산 / 중복 / same-X-different-y 진단
5) 정상 vs 불량 단변량 차이 분석
6) 불량 샘플이 정상분포 어디에 위치하는지 정량화
7) 상관관계 / 후보 변수 랭킹
8) 모델링 전 데이터 구조를 문서화

주의
----
- 모델 학습 없음
- 파생변수 생성 없음
- 1=불량은 Reason 교차표와 함께 데이터로 재확인
"""

from pathlib import Path
import warnings
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import ks_2samp, mannwhitneyu

warnings.filterwarnings("ignore")

# =============================================================================
# 0. 설정
# =============================================================================

BASE_DIR = Path(
    r"C:\Users\Administrator\Desktop\kamp\1. 사출성형기 AI 데이터셋\1. 사출성형기 AI 데이터셋"
)

CN7_WORK_DIR = Path(
    r"C:\Users\Administrator\Desktop\kamp\다시하는 cn7"
)

MAPPING_PATH = CN7_WORK_DIR / "cn7_02_restore" / "cn7_1211_row_mapping.csv"

OUT_DIR = CN7_WORK_DIR / "cn7_03_eda"
PLOT_DIR = OUT_DIR / "plots"

OUT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_NAMES = ["moldset_labeled.csv", "moldset labeled.csv"]
TARGET_NAMES = ["moldset_labeled_cn7.csv", "moldset labeled cn7.csv"]

TARGET_COL = "PassOrFail"

INDEX_LIKE = {"unnamed: 0", "index", "_id", "id"}

# =============================================================================
# 유틸
# =============================================================================

def banner(title):
    print("\n" + "=" * 105)
    print(title)
    print("=" * 105)


def norm(x):
    return str(x).strip().lower()


def find_file(base_dir, names):
    names = {x.lower() for x in names}
    for p in base_dir.rglob("*"):
        if p.is_file() and p.name.lower() in names:
            return p
    return None


def read_csv_flexible(path):
    last = None
    for enc in [None, "utf-8-sig", "utf-8", "cp949", "euc-kr"]:
        try:
            if enc is None:
                return pd.read_csv(path)
            return pd.read_csv(path, encoding=enc)
        except Exception as e:
            last = e
    raise RuntimeError(f"CSV 읽기 실패: {path}\n{last}")


def find_col(df, names):
    m = {norm(c): c for c in df.columns}
    for x in names:
        if norm(x) in m:
            return m[norm(x)]
    return None


def extract_side(series):
    s = series.astype(str).str.upper()
    out = pd.Series("UNKNOWN", index=series.index, dtype="object")
    out[s.str.contains(r"\bRH\b", regex=True, na=False)] = "RH"
    out[s.str.contains(r"\bLH\b", regex=True, na=False)] = "LH"
    return out


def safe_div(a, b):
    return np.nan if b == 0 else a / b


def effect_size_smd(normal, defect):
    """
    Standardized mean difference (pooled SD).
    표본 수가 매우 작은 경우 해석은 보조적으로만 사용.
    """
    n0 = len(normal)
    n1 = len(defect)

    if n0 < 2 or n1 < 2:
        return np.nan

    v0 = normal.var(ddof=1)
    v1 = defect.var(ddof=1)

    pooled = np.sqrt(((n0 - 1) * v0 + (n1 - 1) * v1) / (n0 + n1 - 2))

    if not np.isfinite(pooled) or pooled < 1e-12:
        return np.nan

    return (defect.mean() - normal.mean()) / pooled


def robust_shift(normal, defect):
    """
    정상군 IQR 기준 defect median 이동량.
    """
    q25 = normal.quantile(.25)
    q75 = normal.quantile(.75)
    iqr = q75 - q25

    if abs(iqr) < 1e-12:
        return np.nan

    return (defect.median() - normal.median()) / iqr


def percent_outside_normal_range(normal, defect):
    lo = normal.min()
    hi = normal.max()

    if len(defect) == 0:
        return np.nan

    return ((defect < lo) | (defect > hi)).mean()


def percent_outside_normal_iqr_fence(normal, defect):
    q25 = normal.quantile(.25)
    q75 = normal.quantile(.75)
    iqr = q75 - q25

    lo = q25 - 1.5 * iqr
    hi = q75 + 1.5 * iqr

    if len(defect) == 0:
        return np.nan

    return ((defect < lo) | (defect > hi)).mean()


# =============================================================================
# 1. 로드 + raw 1211 복원
# =============================================================================

banner("CN7 03 - 원본 물리단위 EDA")

source_path = find_file(BASE_DIR, SOURCE_NAMES)
scaled_path = find_file(BASE_DIR, TARGET_NAMES)

if source_path is None:
    raise FileNotFoundError("moldset_labeled.csv를 찾지 못했습니다.")

if scaled_path is None:
    raise FileNotFoundError("moldset_labeled_cn7.csv를 찾지 못했습니다.")

if not MAPPING_PATH.exists():
    raise FileNotFoundError(
        f"CN7_02 mapping 파일이 없습니다:\n{MAPPING_PATH}"
    )

source_full = read_csv_flexible(source_path)
scaled = read_csv_flexible(scaled_path)
mapping = pd.read_csv(MAPPING_PATH)

print("source_full:", source_full.shape)
print("scaled     :", scaled.shape)
print("mapping    :", mapping.shape)

mapping = mapping.sort_values("target_pos").reset_index(drop=True)

# target row 순서에 맞춘 raw source
source_indices = mapping["source_row_index"].astype(int).to_numpy()
raw = source_full.loc[source_indices].copy().reset_index(drop=False)
raw = raw.rename(columns={"index": "__source_row_index__"})

# scaled target row 순서 맞춤
target_indices = mapping["target_row_index"].astype(int).to_numpy()
scaled_ordered = scaled.loc[target_indices].copy().reset_index(drop=False)
scaled_ordered = scaled_ordered.rename(
    columns={"index": "__scaled_row_index__"}
)

if len(raw) != len(scaled_ordered):
    raise RuntimeError("raw/scaled 행 수가 일치하지 않습니다.")

# scaled target을 raw target과 비교
if TARGET_COL not in raw.columns or TARGET_COL not in scaled_ordered.columns:
    raise RuntimeError("PassOrFail 컬럼을 찾지 못했습니다.")

raw_target = pd.to_numeric(raw[TARGET_COL], errors="coerce")
scaled_target = pd.to_numeric(scaled_ordered[TARGET_COL], errors="coerce")

target_same_rate = (raw_target.to_numpy() == scaled_target.to_numpy()).mean()

print("Target raw/scaled same rate:", target_same_rate)

if target_same_rate < 0.999:
    raise RuntimeError(
        "CN7_02 결과와 달리 target이 정확히 매칭되지 않습니다."
    )

# Side
part_name_col = find_col(raw, ["PART_NAME"])
if part_name_col is not None:
    raw["Side"] = extract_side(raw[part_name_col])
else:
    raw["Side"] = "UNKNOWN"

# 시간
timestamp_col = find_col(raw, ["TimeStamp"])
if timestamp_col is not None:
    raw["TimeStamp_dt"] = pd.to_datetime(
        raw[timestamp_col],
        errors="coerce",
    )

# clean target 명시
raw["IsDefect"] = raw_target.astype(int)

# 원본 물리단위 데이터 저장
raw.to_csv(
    OUT_DIR / "cn7_train_raw_1211.csv",
    index=False,
    encoding="utf-8-sig",
)

print("raw restored:", raw.shape)

# =============================================================================
# 2. Target 의미 검증
# =============================================================================

banner("1) Target 의미 검증")

print("\nPassOrFail counts:")
print(raw[TARGET_COL].value_counts(dropna=False).sort_index().to_string())

reason_col = find_col(raw, ["Reason"])

if reason_col is not None:
    print("\nReason counts:")
    print(
        raw[reason_col]
        .fillna("<NA>")
        .value_counts(dropna=False)
        .to_string()
    )

    print("\nReason x PassOrFail:")
    reason_target = pd.crosstab(
        raw[reason_col].fillna("<NA>"),
        raw[TARGET_COL],
        margins=True,
    )
    print(reason_target.to_string())

    reason_target.to_csv(
        OUT_DIR / "reason_by_target.csv",
        encoding="utf-8-sig",
    )
else:
    print("Reason 컬럼 없음")

# =============================================================================
# 3. 기본 구조
# =============================================================================

banner("2) Target / Side / 생산 metadata")

# Target
target_summary = (
    raw[TARGET_COL]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis(TARGET_COL)
    .reset_index(name="count")
)
target_summary["rate"] = target_summary["count"] / len(raw)

print("\nTarget:")
print(target_summary.to_string(index=False))

target_summary.to_csv(
    OUT_DIR / "target_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

# Side
side_target = pd.crosstab(
    raw["Side"],
    raw[TARGET_COL],
    margins=True,
)

print("\nSide x Target:")
print(side_target.to_string())

side_target.to_csv(
    OUT_DIR / "side_by_target.csv",
    encoding="utf-8-sig",
)

side_rate_rows = []

for side, g in raw.groupby("Side"):
    n = len(g)
    d = int((g[TARGET_COL] == 1).sum())

    side_rate_rows.append({
        "Side": side,
        "n": n,
        "target_1": d,
        "target_1_rate": safe_div(d, n),
    })

side_rate_df = pd.DataFrame(side_rate_rows)
side_rate_df.to_csv(
    OUT_DIR / "side_target_rates.csv",
    index=False,
    encoding="utf-8-sig",
)

print("\nSide target-1 rate:")
print(side_rate_df.to_string(index=False))

# 날짜
plan_col = find_col(raw, ["PART_FACT_PLAN_DATE"])
serial_col = find_col(raw, ["PART_FACT_SERIAL"])

if plan_col:
    print("\nPART_FACT_PLAN_DATE x Target:")
    plan_target = pd.crosstab(
        raw[plan_col],
        raw[TARGET_COL],
        margins=True,
    )
    print(plan_target.to_string())
    plan_target.to_csv(
        OUT_DIR / "plan_date_by_target.csv",
        encoding="utf-8-sig",
    )

if serial_col:
    print("\nPART_FACT_SERIAL x Target:")
    serial_target = pd.crosstab(
        raw[serial_col],
        raw[TARGET_COL],
        margins=True,
    )
    print(serial_target.to_string())
    serial_target.to_csv(
        OUT_DIR / "serial_by_target.csv",
        encoding="utf-8-sig",
    )

# =============================================================================
# 4. 센서 컬럼 확정
# =============================================================================

banner("3) 공정 센서 컬럼")

scaled_map = {norm(c): c for c in scaled.columns}

sensor_cols = []

for c in raw.columns:
    if norm(c) in scaled_map:
        if c == TARGET_COL:
            continue
        if norm(c) in INDEX_LIKE:
            continue
        if pd.api.types.is_numeric_dtype(raw[c]):
            sensor_cols.append(c)

print("센서 수:", len(sensor_cols))
print(sensor_cols)

# =============================================================================
# 5. 센서 품질 진단
# =============================================================================

banner("4) 상수 / unique / 분산")

quality_rows = []

for c in sensor_cols:
    s = pd.to_numeric(raw[c], errors="coerce")

    quality_rows.append({
        "feature": c,
        "dtype": str(raw[c].dtype),
        "n_unique": int(s.nunique(dropna=False)),
        "missing": int(s.isna().sum()),
        "missing_rate": float(s.isna().mean()),
        "mean": s.mean(),
        "std": s.std(ddof=1),
        "cv_abs": (
            abs(s.std(ddof=1) / s.mean())
            if abs(s.mean()) > 1e-12
            else np.nan
        ),
        "min": s.min(),
        "q01": s.quantile(.01),
        "q25": s.quantile(.25),
        "median": s.median(),
        "q75": s.quantile(.75),
        "q99": s.quantile(.99),
        "max": s.max(),
        "constant": s.nunique(dropna=False) <= 1,
    })

quality_df = pd.DataFrame(quality_rows)

print(
    quality_df[
        [
            "feature",
            "n_unique",
            "missing",
            "mean",
            "std",
            "min",
            "max",
            "constant",
        ]
    ]
    .round(4)
    .to_string(index=False)
)

quality_df.to_csv(
    OUT_DIR / "sensor_quality_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

constant_cols = quality_df.loc[
    quality_df["constant"],
    "feature",
].tolist()

print("\n상수 컬럼:", constant_cols)

# =============================================================================
# 6. 중복 구조
# =============================================================================

banner("5) 동일 공정조건 반복 구조")

usable_sensor_cols = [
    c for c in sensor_cols
    if c not in constant_cols
]

dup_mask = raw.duplicated(
    subset=usable_sensor_cols,
    keep=False,
)

print(
    "동일 sensor vector에 포함되는 행:",
    int(dup_mask.sum()),
    "/",
    len(raw),
)

grouped = (
    raw.groupby(
        usable_sensor_cols,
        dropna=False,
        sort=False,
    )
    .agg(
        rows=(TARGET_COL, "size"),
        target_nunique=(TARGET_COL, "nunique"),
        target_1=(TARGET_COL, lambda x: int((x == 1).sum())),
        side_nunique=("Side", "nunique"),
        sides=("Side", lambda x: "|".join(sorted(set(map(str, x))))),
    )
    .reset_index()
)

dup_groups = grouped[grouped["rows"] > 1].copy()
conflict_groups = grouped[grouped["target_nunique"] > 1].copy()

print("중복 그룹 수:", len(dup_groups))
print("same-X / different-target 그룹:", len(conflict_groups))

dup_groups.to_csv(
    OUT_DIR / "duplicate_sensor_groups.csv",
    index=False,
    encoding="utf-8-sig",
)

conflict_groups.to_csv(
    OUT_DIR / "same_X_different_target_groups.csv",
    index=False,
    encoding="utf-8-sig",
)

# conflict 원본 행 상세
if len(conflict_groups):
    conflict_keys = conflict_groups[usable_sensor_cols]

    detailed = raw.merge(
        conflict_keys,
        on=usable_sensor_cols,
        how="inner",
    )

    detail_cols = [
        c for c in [
            "__source_row_index__",
            timestamp_col,
            plan_col,
            serial_col,
            part_name_col,
            "Side",
            TARGET_COL,
            reason_col,
        ]
        if c is not None and c in detailed.columns
    ] + usable_sensor_cols

    detailed[detail_cols].to_csv(
        OUT_DIR / "same_X_different_target_rows.csv",
        index=False,
        encoding="utf-8-sig",
    )

# =============================================================================
# 7. 정상 vs 불량 단변량 분석
# =============================================================================

banner("6) 정상 vs 불량 단변량 분석")

normal = raw[raw[TARGET_COL] == 0].copy()
defect = raw[raw[TARGET_COL] == 1].copy()

print("normal:", len(normal))
print("target=1:", len(defect))

univariate_rows = []

for c in usable_sensor_cols:
    a = pd.to_numeric(normal[c], errors="coerce").dropna()
    b = pd.to_numeric(defect[c], errors="coerce").dropna()

    if len(a) == 0 or len(b) == 0:
        continue

    try:
        ks = ks_2samp(a, b, alternative="two-sided", mode="auto")
        ks_stat = float(ks.statistic)
        ks_p = float(ks.pvalue)
    except Exception:
        ks_stat = np.nan
        ks_p = np.nan

    try:
        mw = mannwhitneyu(
            a,
            b,
            alternative="two-sided",
        )
        mw_stat = float(mw.statistic)
        mw_p = float(mw.pvalue)
    except Exception:
        mw_stat = np.nan
        mw_p = np.nan

    smd = effect_size_smd(a, b)
    rshift = robust_shift(a, b)

    univariate_rows.append({
        "feature": c,

        "normal_mean": a.mean(),
        "defect_mean": b.mean(),
        "mean_diff": b.mean() - a.mean(),
        "abs_mean_diff": abs(b.mean() - a.mean()),

        "normal_median": a.median(),
        "defect_median": b.median(),
        "median_diff": b.median() - a.median(),

        "normal_std": a.std(ddof=1),
        "defect_std": b.std(ddof=1),

        "SMD": smd,
        "Abs_SMD": abs(smd) if pd.notna(smd) else np.nan,

        "Robust_Shift_IQR": rshift,
        "Abs_Robust_Shift_IQR": (
            abs(rshift) if pd.notna(rshift) else np.nan
        ),

        "KS_stat": ks_stat,
        "KS_pvalue": ks_p,
        "MW_stat": mw_stat,
        "MW_pvalue": mw_p,

        "Defect_Outside_Normal_Range_Rate":
            percent_outside_normal_range(a, b),

        "Defect_Outside_Normal_IQR_Fence_Rate":
            percent_outside_normal_iqr_fence(a, b),

        "normal_n": len(a),
        "defect_n": len(b),
    })

uni_df = pd.DataFrame(univariate_rows)

# 데이터가 17개 불량뿐이므로 p-value 단독 정렬은 하지 않음
uni_df["Composite_Exploration_Score"] = (
    uni_df["Abs_SMD"].fillna(0)
    + uni_df["KS_stat"].fillna(0)
    + uni_df["Abs_Robust_Shift_IQR"].clip(upper=10).fillna(0) * 0.5
    + uni_df["Defect_Outside_Normal_IQR_Fence_Rate"].fillna(0)
)

uni_df = uni_df.sort_values(
    "Composite_Exploration_Score",
    ascending=False,
)

print(
    uni_df[
        [
            "feature",
            "normal_mean",
            "defect_mean",
            "SMD",
            "KS_stat",
            "KS_pvalue",
            "Robust_Shift_IQR",
            "Defect_Outside_Normal_Range_Rate",
            "Defect_Outside_Normal_IQR_Fence_Rate",
            "Composite_Exploration_Score",
        ]
    ]
    .round(4)
    .to_string(index=False)
)

uni_df.to_csv(
    OUT_DIR / "normal_vs_defect_univariate.csv",
    index=False,
    encoding="utf-8-sig",
)

# =============================================================================
# 8. Side별 정상 vs 불량
# =============================================================================

banner("7) Side별 정상 vs 불량")

side_uni_all = []

for side in sorted(raw["Side"].dropna().unique()):
    g = raw[raw["Side"] == side]

    g0 = g[g[TARGET_COL] == 0]
    g1 = g[g[TARGET_COL] == 1]

    print(
        f"{side}: n={len(g)}, normal={len(g0)}, target1={len(g1)}"
    )

    if len(g1) < 2:
        continue

    for c in usable_sensor_cols:
        a = pd.to_numeric(g0[c], errors="coerce").dropna()
        b = pd.to_numeric(g1[c], errors="coerce").dropna()

        if len(a) == 0 or len(b) == 0:
            continue

        try:
            ks = ks_2samp(a, b)
            ks_stat = float(ks.statistic)
            ks_p = float(ks.pvalue)
        except Exception:
            ks_stat = np.nan
            ks_p = np.nan

        smd = effect_size_smd(a, b)

        side_uni_all.append({
            "Side": side,
            "feature": c,
            "normal_n": len(a),
            "defect_n": len(b),
            "normal_mean": a.mean(),
            "defect_mean": b.mean(),
            "SMD": smd,
            "Abs_SMD": abs(smd) if pd.notna(smd) else np.nan,
            "KS_stat": ks_stat,
            "KS_pvalue": ks_p,
        })

side_uni_df = pd.DataFrame(side_uni_all)

if len(side_uni_df):
    side_uni_df = side_uni_df.sort_values(
        ["Side", "Abs_SMD"],
        ascending=[True, False],
    )

    side_uni_df.to_csv(
        OUT_DIR / "normal_vs_defect_by_side.csv",
        index=False,
        encoding="utf-8-sig",
    )

# =============================================================================
# 9. 센서 상관관계
# =============================================================================

banner("8) 센서 상관관계")

corr = raw[usable_sensor_cols].corr(method="spearman")

corr.to_csv(
    OUT_DIR / "sensor_spearman_correlation.csv",
    encoding="utf-8-sig",
)

pairs_corr = []

for i, a in enumerate(usable_sensor_cols):
    for j in range(i + 1, len(usable_sensor_cols)):
        b = usable_sensor_cols[j]
        v = corr.loc[a, b]

        pairs_corr.append({
            "feature_1": a,
            "feature_2": b,
            "spearman": v,
            "abs_spearman": abs(v),
        })

corr_pair_df = (
    pd.DataFrame(pairs_corr)
    .sort_values("abs_spearman", ascending=False)
)

print("\n상관 높은 센서 pair:")
print(corr_pair_df.head(20).round(4).to_string(index=False))

corr_pair_df.to_csv(
    OUT_DIR / "sensor_correlation_pairs.csv",
    index=False,
    encoding="utf-8-sig",
)

# =============================================================================
# 10. 불량 발생 순서 / 시간
# =============================================================================

banner("9) 불량 발생 시간 구조")

if "TimeStamp_dt" in raw.columns:
    order_df = raw.sort_values("TimeStamp_dt").copy()
    order_df["time_order"] = np.arange(len(order_df))

    defect_rows = order_df[
        order_df[TARGET_COL] == 1
    ].copy()

    keep = [
        c for c in [
            "time_order",
            "__source_row_index__",
            "TimeStamp_dt",
            plan_col,
            serial_col,
            "Side",
            TARGET_COL,
            reason_col,
        ]
        if c is not None and c in defect_rows.columns
    ]

    print(defect_rows[keep].to_string(index=False))

    defect_rows.to_csv(
        OUT_DIR / "defect_rows_time_order.csv",
        index=False,
        encoding="utf-8-sig",
    )

    # 날짜별
    order_df["Date"] = order_df["TimeStamp_dt"].dt.date

    date_target = pd.crosstab(
        order_df["Date"],
        order_df[TARGET_COL],
        margins=True,
    )

    print("\n날짜 x Target:")
    print(date_target.to_string())

    date_target.to_csv(
        OUT_DIR / "date_by_target.csv",
        encoding="utf-8-sig",
    )

# =============================================================================
# 11. 핵심 그래프
# =============================================================================

banner("10) 그래프 저장")

# Target distribution
fig = plt.figure(figsize=(6, 4))
raw[TARGET_COL].value_counts().sort_index().plot(kind="bar")
plt.title("CN7 PassOrFail Distribution")
plt.xlabel("PassOrFail")
plt.ylabel("Count")
plt.tight_layout()
plt.savefig(
    PLOT_DIR / "target_distribution.png",
    dpi=150,
)
plt.close(fig)

# Side target rate
if len(side_rate_df):
    fig = plt.figure(figsize=(6, 4))
    plt.bar(
        side_rate_df["Side"],
        side_rate_df["target_1_rate"],
    )
    plt.title("Target=1 Rate by Side")
    plt.xlabel("Side")
    plt.ylabel("Rate")
    plt.tight_layout()
    plt.savefig(
        PLOT_DIR / "target1_rate_by_side.png",
        dpi=150,
    )
    plt.close(fig)

# Top univariate boxplots
top_features = uni_df.head(10)["feature"].tolist()

for rank, c in enumerate(top_features, 1):
    fig = plt.figure(figsize=(6, 4))

    values0 = pd.to_numeric(
        raw.loc[raw[TARGET_COL] == 0, c],
        errors="coerce",
    ).dropna()

    values1 = pd.to_numeric(
        raw.loc[raw[TARGET_COL] == 1, c],
        errors="coerce",
    ).dropna()

    plt.boxplot(
        [values0, values1],
        tick_labels=["0", "1"],
        showfliers=True,
    )
    plt.title(c)
    plt.xlabel("PassOrFail")
    plt.ylabel(c)
    plt.tight_layout()

    safe_name = c.replace("/", "_").replace("\\", "_")

    plt.savefig(
        PLOT_DIR / f"top_{rank:02d}_{safe_name}.png",
        dpi=150,
    )
    plt.close(fig)

print("그래프 저장:", PLOT_DIR)

# =============================================================================
# 12. 자동 요약
# =============================================================================

banner("11) 자동 EDA 요약")

summary = []

n = len(raw)
n1 = int((raw[TARGET_COL] == 1).sum())
n0 = int((raw[TARGET_COL] == 0).sum())

summary.append(
    f"- 전체 {n}행: target0={n0}, target1={n1}, target1_rate={n1/n:.3%}"
)

if reason_col is not None:
    reason_nonnull = raw[
        raw[reason_col].notna()
        & (raw[reason_col].astype(str).str.strip() != "")
    ]
    summary.append(
        f"- Reason 기록 행: {len(reason_nonnull)}"
    )

if len(side_rate_df):
    for _, r in side_rate_df.iterrows():
        summary.append(
            f"- Side {r['Side']}: n={int(r['n'])}, "
            f"target1={int(r['target_1'])}, "
            f"rate={r['target_1_rate']:.3%}"
        )

summary.append(
    f"- 상수 센서: {constant_cols}"
)
summary.append(
    f"- 중복 sensor vector 그룹: {len(dup_groups)}"
)
summary.append(
    f"- same-X/different-target 그룹: {len(conflict_groups)}"
)

if len(uni_df):
    summary.append(
        "- 단변량 탐색 상위 10개: "
        + ", ".join(uni_df.head(10)["feature"].tolist())
    )

if "TimeStamp_dt" in raw.columns:
    summary.append(
        f"- 시간 범위: "
        f"{raw['TimeStamp_dt'].min()} ~ {raw['TimeStamp_dt'].max()}"
    )

for x in summary:
    print(x)

with open(
    OUT_DIR / "eda_summary.txt",
    "w",
    encoding="utf-8",
) as f:
    f.write("\n".join(summary))

# =============================================================================
# 완료
# =============================================================================

banner("완료")

print("결과 폴더:", OUT_DIR)
print()
print("핵심 파일:")
print("  1) cn7_train_raw_1211.csv")
print("  2) reason_by_target.csv")
print("  3) side_by_target.csv")
print("  4) sensor_quality_summary.csv")
print("  5) same_X_different_target_groups.csv")
print("  6) same_X_different_target_rows.csv")
print("  7) normal_vs_defect_univariate.csv")
print("  8) normal_vs_defect_by_side.csv")
print("  9) defect_rows_time_order.csv")
print(" 10) sensor_correlation_pairs.csv")
print(" 11) eda_summary.txt")
print()
print(
    "다음 단계는 이 EDA 결과를 보고 "
    "Validation 방식을 결정하는 것입니다."
)



CN7 03 - 원본 물리단위 EDA
source_full: (2607, 47)
scaled     : (1211, 26)
mapping    : (1211, 9)
Target raw/scaled same rate: 1.0
raw restored: (1211, 51)

1) Target 의미 검증

PassOrFail counts:
PassOrFail
0    1194
1      17

Reason counts:
Reason
<NA>    1194
가스        13
미성형        4

Reason x PassOrFail:
PassOrFail     0   1   All
Reason                    
<NA>        1194   0  1194
가스             0  13    13
미성형            0   4     4
All         1194  17  1211

2) Target / Side / 생산 metadata

Target:
 PassOrFail  count     rate
          0   1194 0.985962
          1     17 0.014038

Side x Target:
PassOrFail     0   1   All
Side                      
LH           601   4   605
RH           593  13   606
All         1194  17  1211

Side target-1 rate:
Side   n  target_1  target_1_rate
  LH 605         4       0.006612
  RH 606        13       0.021452

PART_FACT_PLAN_DATE x Target:
PassOrFail                 0   1   All
PART_FACT_PLAN_DATE                   
2020-10-16 오전 12:00:00   35